In [8]:
# -*- coding: utf-8 -*-

#Script developed by Justine Hansen 
#Original script: https://github.com/netneurolab/hansen_receptors/blob/main/code/https://github.com/netneurolab/hansen_receptors/blob/main/code/disease.py
#Reference: Hansen JY et al. 'Mapping neurotransmitter systems to the structural and functional organization of the human neocortex' Nat Neurosci 2022

"""
Autoradiography MIND
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import patches
import seaborn as sns
from netneurotools import datasets, metrics, stats, plotting
from scipy.stats import zscore, pearsonr
from scipy.optimize import curve_fit
from matplotlib.colors import ListedColormap
from scipy.spatial.distance import squareform, pdist
from sklearn.linear_model import LinearRegression
from sklearn.utils.validation import check_random_state
from sklearn.decomposition import PCA
from nilearn.datasets import fetch_atlas_schaefer_2018
import pyls
from statsmodels.stats.multitest import multipletests

In [9]:
def make_autorad_schaefer100(autorad_zilles44):
    
    autorad_schaefer100 = np.zeros((50, autorad_zilles44.shape[1]))
    # mapping between zilles and schaefer (done manually)
    zilles44_to_schaefer100 = \
        [[21],  # area '36' // 0
         [9],   # area 'V3v' // 1
         [6],   # area 'V2v' // 2
         [4, 6],   # area 'V1', 'V2v'  // 3
         [4, 5, 6],   # area 'V1', 'V2v', 'V2d' // 4
         [4],   # area 'V1' // 5
         [np.nan],  # // 6
         [8, 7],   # area 'V3d', 'V3a' // 7
         [8],   # area 'V3d' // 8
         [17],  # area '42' // 9
         [16],  # area '41' // 10
         [np.nan], # // 11
         [0, 3, 2],  # area '1', '3b', '3a' \\ 12
         [0, 3, 2],  # area '1', '3b', '3a' \\ 13
         [0, 3, 2],  # area '1', '3b', '3a'\\ 14
         [14],  # area '37L' \\ 15
         [27, 1],  # area 'PFt', '2' \\ 16
         [np.nan],  # // 17
         [1],  # area '2' // 18
         [24],  # area '5M' // 19
         [23],  # area '5L' // 20
         [0, 34],  # area '1', '6' // 21
         [35],  # area '8' // 22
         [28, 27],  # area 'PFm', 'PFt' // 23
         [np.nan],  # insula // 24
         [41],  # area '45' // 25
         [36],  # area '9' // 26
         [29],  # area '24' // 27
         [24],  # area '5M' // 28
         [35],  # area '8' // 29
         [39],  # area '11' // 30
         [22],  # area '38' // 31
         [18],  # area '20' // 32
         [28, 27],  # area 'PFm' 'PFt' // 33
         [42],  # area '46' // 34
         [7],  # area 'V3A' // 35
         [31],  # area '23' (and 31?) // 36
         [20, 19],  # area '22', '21' // 37
         [19, 20],  # area '21', '22' // 38
         [19, 20],  # area '21', '22' // 39
         [25, 26],  # area 'PGa', 'PGp' // 40
         [43],  # area '47' // 41
         [41, 43, 42],  # area '45', '47', '46' // 42
         [29, 30, 38],  # area '24', '32', '10M' // 43
         [37],  # area '10L' // 44
         [36],  # area '9' // 45
         [35, 36],  # area '8', '9' // 46
         [35],  # area '8' // 47
         [32],  # area '31' // 48
         [32]]  # area '31' // 49 

    for n in range(autorad_schaefer100.shape[0]):
        if np.isnan(zilles44_to_schaefer100[n][0]):
            autorad_schaefer100[n, :] = np.nan
        elif len(zilles44_to_schaefer100[n]) == 1:
            autorad_schaefer100[n, :] = autorad_zilles44[zilles44_to_schaefer100[n], :]
        elif len(zilles44_to_schaefer100[n]) > 1:
            autorad_schaefer100[n, :] = np.mean(autorad_zilles44[zilles44_to_schaefer100[n], :], axis=0).T

    return autorad_schaefer100


In [10]:
def make_autorad_cammoun033(autorad_zilles44):
    # region indeces associated with more than one dk region
    duplicate = [20, 21, 28, 29, 30, 32, 34, 39]

    # mapping from 44 brodmann areas + 7 duplicate regions to dk left hem
    # manually done, comparing anatomical regions to one another
    # originally written in matlab and too lazy to change indices hence the -1
    # the index refers to the cammoun scale033 structure name
    mapping = np.array([57, 57, 57, 57, 63, 62, 65, 62, 64, 65, 64, 66, 66,
                        66, 66, 66, 74, 74, 70, 71, 72, 73, 67, 68, 69, 52,
                        52, 60, 60, 58, 58, 59, 53, 54, 53, 54, 55, 56, 61,
                        51, 51, 50, 49, 49, 44, 44, 45, 42, 47, 46, 48, 43])
    mapping = mapping - min(mapping)  # python indexing

    rep = np.ones((autorad_zilles44.shape[0], ), dtype=int)  # duplicate regions
    rep[duplicate] = 2
    autorad_zilles44 = np.repeat(autorad_zilles44, rep, 0)

    # convert to dk
    n_dknodes = max(mapping) + 1  # number of nodes in dk atlas (left hem only)

    u = np.unique(mapping)
    autorad_cammoun033 = np.zeros((n_dknodes, autorad_zilles44.shape[1]))
    for i in range(len(u)):
        if sum(mapping == u[i]) > 1:
            autorad_cammoun033[u[i], :] = np.mean(autorad_zilles44[mapping == u[i], :], axis=0)
        else:
            autorad_cammoun033[u[i], :] = autorad_zilles44[mapping == u[i], :]
    return autorad_cammoun033

In [11]:
def get_reg_r_sq(X, y):
    lin_reg = LinearRegression()
    lin_reg.fit(X, y)
    yhat = lin_reg.predict(X)
    SS_Residual = sum((y - yhat) ** 2)
    SS_Total = sum((y - np.mean(y)) ** 2)
    r_squared = 1 - (float(SS_Residual)) / SS_Total
    adjusted_r_squared = 1 - (1 - r_squared) * \
        (len(y) - 1) / (len(y) - X.shape[1] - 1)
    return adjusted_r_squared, SS_Residual


def get_perm_p(emp, null):
    return (1 + sum(abs(null - np.mean(null))
                    > abs(emp - np.mean(null)))) / (len(null) + 1)


def corr_perm(x, y, perms, nperms):
    rho = pearsonr(x, y)[0]
    null = np.zeros((nperms, ))
    for i in range(nperms):
        null[i] = pearsonr(x, y[perms[:, i]])[0]
    pval = get_perm_p(rho, null)
    return rho, pval
    

In [12]:
def match_length_degree_distribution(data, eu_distance, nbins=10, nswap=None, seed=None):
    """
    Takes a weighted, symmetric connectivity matrix `data` and Euclidean/fiber 
    length matrix `distance` and generates a randomized network with:
        1. exactly the same degree sequence
        2. approximately the same edge length distribution
        3. exactly the same edge weight distribution
        4. approximately the same weight-length relationship

    Parameters
    ----------
    data : (N, N) array-like
        weighted or binary symmetric connectivity matrix.
    distance : (N, N) array-like
        symmetric distance matrix.
    nbins : int
        number of distance bins (edge length matrix is performed by swapping
        connections in the same bin). Default = 10.
    nswap : int
        total number of edge swaps to perform. Recommended = nnodes * 20.
        Default = None.

    Returns
    -------
    data : (N, N) array-like
        binary rewired matrix
    W : (N, N) array-like
        weighted rewired matrix
        
    Reference
    ---------
    Betzel, R. F., Bassett, D. S. (2018) Specificity and robustness of long-distance
    connections in weighted, interareal connectomes. PNAS.

    """
    rs = check_random_state(seed)

    nnodes = len(data)             # number of nodes
    
    if nswap is None:
        nswap = nnodes*20          # set default number of swaps
    
    mask = data != 0               # nonzero elements
    mask = np.triu(mask, 1)        # keep upper triangle only
    weights = data[mask]           # values of edge weights
    distances = eu_distance[mask]  # values of edge lengths
    Jdx = np.argsort(distances)    # indices to sort distances in ascending order
    
    bins = np.linspace(min(eu_distance[eu_distance != 0]),
                       max(eu_distance[eu_distance != 0]),
                       nbins+1)  # length/distance of bins
    bins[-1] += 1
    B = np.zeros((nnodes, nnodes, nbins))  # initiate 3D stack of bins
    for k in range(nbins):
        # element is k+1 if the distance falls within the bin, 0 otherwise
        B[:, :, k] = np.logical_and(eu_distance >= bins[k],
                                   eu_distance < bins[k + 1]) * (k + 1)
    # matrix of distance bins
    Bsum = np.sum(B, axis=2)
    
    tmp = np.triu((data != 0)*Bsum, 1)
    row_idx, col_idx = tmp.nonzero()  # indices of edges
    vals = tmp[row_idx, col_idx]
    nedges = len(row_idx)  # number of edges
    iswap = 0             # swap counter
    
    while iswap < nswap:
        myEdge = rs.randint(nedges)   # get a random edge index
        myEdge_row = row_idx[myEdge]  # row idx of edge
        myEdge_col = col_idx[myEdge]  # col idx of edge
        myEdge_bin = vals[myEdge]     # bin of edge
        
        # get indices that can be swapped
        indkeep = (row_idx != myEdge_row) & (row_idx != myEdge_col) \
                  & (col_idx != myEdge_row) & (col_idx != myEdge_col)

        row_idx_keep = row_idx[indkeep]
        col_idx_keep = col_idx[indkeep]
        
        bins_keep = vals[indkeep]  # bins of possible swaps
        
        edge_row = myEdge_row*nnodes + row_idx_keep # edge indices
        edge_row_bins = Bsum[np.unravel_index(edge_row, Bsum.shape)] # matlab-style linear indexing
        edge_col = myEdge_col*nnodes + col_idx_keep # other set of edge indices
        edge_col_bins = Bsum[np.unravel_index(edge_col, Bsum.shape)] 
        
        # get good list of indices
        idx1 = np.logical_and(myEdge_bin == edge_row_bins,
                              bins_keep == edge_col_bins)
        # get other set of good indices
        idx2 = np.logical_and(myEdge_bin == edge_col_bins,
                              bins_keep == edge_row_bins)
        # full set
        goodidx = np.logical_or(idx1, idx2)
        
        # update the indices to keep
        row_idx_keep = row_idx_keep[goodidx]
        col_idx_keep = col_idx_keep[goodidx]
        
        # update the edge indices
        edge_row = myEdge_row*nnodes + row_idx_keep
        edge_col = myEdge_col*nnodes + col_idx_keep
        
        data_row = data[np.unravel_index(edge_row, data.shape)]
        data_col = data[np.unravel_index(edge_col, data.shape)]
        
        # find missing edges
        ind = np.where(np.logical_and(data_row == 0,
                                      data_col == 0).astype(int))[0]
        
        if len(ind) > 0:  # if there is a missing edge
            
            # choose a random swap
            random_swap = ind[rs.randint(len(ind))]
            
            # do the swap
            row_idx_keep = row_idx_keep[random_swap]
            col_idx_keep = col_idx_keep[random_swap]
            
            data[myEdge_row, myEdge_col] = 0
            data[myEdge_col, myEdge_row] = 0
            
            data[row_idx_keep, col_idx_keep] = 0
            data[col_idx_keep, row_idx_keep] = 0
            
            data[myEdge_row, row_idx_keep] = 1
            data[row_idx_keep, myEdge_row] = 1
            
            data[myEdge_col, col_idx_keep] = 1
            data[col_idx_keep, myEdge_col] = 1
            
            other_edge = np.where(indkeep)[0]
            other_edge = other_edge[goodidx]
            other_edge = other_edge[random_swap]
            
            row_idx[myEdge] = min(myEdge_row, row_idx_keep)
            col_idx[myEdge] = max(myEdge_row, row_idx_keep)
            
            row_idx[other_edge] = min(myEdge_col, col_idx_keep)
            col_idx[other_edge] = max(myEdge_col, col_idx_keep)
            
            vals[myEdge] = Bsum[myEdge_row, row_idx_keep]
            vals[other_edge] = Bsum[myEdge_col, col_idx_keep]
            
            iswap += 1
            if iswap % 100 == 0:
                print(iswap)
    
    d = eu_distance[np.where(np.triu(data, 1))]  # get distances where edges are
    jdx = np.argsort(d)                      # sort distances (ascending)
    W = np.zeros((nnodes, nnodes))           # output matrix
    # add weights
    W[np.where(np.triu(data,1))[0][jdx],
      np.where(np.triu(data,1))[1][jdx]] = weights[Jdx]
    
    return data, W


def exponential(x, a, b, c):
    return a * np.exp(b * x) + c


def regress_dist(x, eu_distance, pars):
    return x - exponential(eu_distance, pars[0], pars[1], pars[2])


def add_hem_for_plotting(data, nnodes_full):
    """
    add in some zeros where data is missing so that plotting is possible
    """
    newdata = np.zeros([nnodes_full, ])
    newdata[:len(data)] = data
    return newdata




In [13]:

"""
set-up
"""

path = '/Users/charlie/Desktop/my_projects/neurotransmitter/github/'

# load atlas
schaefer = fetch_atlas_schaefer_2018(n_rois=100)
annot = datasets.fetch_schaefer2018('fsaverage')['100Parcels7Networks']
nnodes = len(schaefer['labels'])
coords = np.genfromtxt(path+'data/schaefer/coordinates/Schaefer_100_centres.txt')[:, 1:]
hemiid = np.zeros((nnodes, ))
hemiid[:int(nnodes/2)] = 1
nspins = 10000
spins = stats.gen_spinsamples(coords, hemiid, n_rotate=nspins, seed=1234)
eu = squareform(pdist(coords, metric='euclidean'))

# load PET data
PET_data = np.genfromtxt(path+'results/receptor_data_scale100.csv', delimiter=',')

# load autorad data


autorad_zilles44 = np.load(path+'data/autoradiography/ReceptData.npy')
receptor_names = np.load(path+'data/autoradiography/ReceptorNames.npy')
autorad_schaefer100 = make_autorad_schaefer100(autorad_zilles44)
autorad_cammoun033 = make_autorad_cammoun033(autorad_zilles44)
goodidx = np.where(np.isnan(autorad_schaefer100[:, 0]) == False)[0]
badidx = np.where(np.isnan(autorad_schaefer100[:, 0]) == True)[0]
autorad_schaefer100 = zscore(autorad_schaefer100, nan_policy='omit')
nnodes = autorad_schaefer100.shape[0]
autorad_df = pd.DataFrame(data=autorad_schaefer100,
                          index=schaefer['labels'][:50],
                          columns=receptor_names)
perms = np.zeros((len(goodidx), nspins))
for i in range(nspins):
    perms[:, i] = np.random.permutation(len(goodidx))
perms = perms.astype(int)

# load sc fc
sc = np.load(path+'data/schaefer/sc_binary.npy')
sc_weighted = np.load(path+'data/schaefer/sc_weighted.npy')
fc = np.load(path+'data/schaefer/fc_weighted.npy')

# colourmaps
cmap = np.genfromtxt(path+'/data/colourmap.csv', delimiter=',')
cmap_div = ListedColormap(cmap)
cmap_seq = ListedColormap(cmap[128:, :])

# rsn networks for plotting
rsn_mapping = []
for row in range(len(schaefer['labels'])):
    rsn_mapping.append(schaefer['labels'][row].decode('utf-8').split('_')[2])
rsn_mapping = np.array(rsn_mapping)


In [ ]:
###NODE INFRAGRANULAR ######


autorad_zilles44 = np.load(path+'data/autoradiography/ReceptData_I.npy')
receptor_names = np.load(path+'data/autoradiography/ReceptorNames.npy')
autorad_schaefer100 = make_autorad_schaefer100(autorad_zilles44)
autorad_cammoun033 = make_autorad_cammoun033(autorad_zilles44)
goodidx = np.where(np.isnan(autorad_schaefer100[:, 0]) == False)[0]
badidx = np.where(np.isnan(autorad_schaefer100[:, 0]) == True)[0]
autorad_schaefer100 = zscore(autorad_schaefer100, nan_policy='omit')
nnodes = autorad_schaefer100.shape[0]
autorad_df = pd.DataFrame(data=autorad_schaefer100,
                          index=schaefer['labels'][:50],
                          columns=receptor_names)


ct = np.genfromtxt(path+'data/modified_enigma_atrophy_node.csv', delimiter=',')
disorders = ['node_track', 'node_trackon', 'node_yas']

model_metrics = dict([])
model_pval = np.zeros((len(disorders), ))
perms = np.zeros((33, nspins))
for i in range(nspins):
    perms[:, i] = np.random.permutation(33)
perms = perms.astype(int)

for i in range(len(disorders)):
    print(i)
    m, _ = stats.get_dominance_stats(zscore(autorad_cammoun033),
                                     zscore(ct[:33, i]))
    model_metrics[disorders[i]] = m
    # get model pval
    emp, _ = get_reg_r_sq(zscore(autorad_cammoun033),
                          zscore(ct[:33, i]))
    null = np.zeros((nspins, ))
    for s in range(nspins):
        Xnull = autorad_cammoun033[perms[:, s], :]
        null[s], _ = get_reg_r_sq(zscore(Xnull), zscore(ct[:33, i]))
    model_pval[i] = (1 + sum(null > emp))/(nspins + 1)

dominance = np.zeros((len(disorders), len(receptor_names)))

for i in range(len(model_metrics)):
    tmp = model_metrics[disorders[i]]
    dominance[i, :] = tmp["total_dominance"]
np.save(path+'results/dominance_enigma_autorad_node_IG.npy', dominance)

plt.ion()
plt.figure()
plt.bar(np.arange(len(disorders)), np.sum(dominance, axis=1),
        tick_label=disorders)
plt.xticks(rotation='vertical')
plt.tight_layout()
plt.savefig(path+'figures/schaefer100/bar_dominance_enigma_aut_node_IG_6colors.eps')

model_pval = multipletests(model_pval, method='fdr_bh')[1]
dominance[np.where(model_pval >= 0.05)[0], :] = 0

plt.ion()
plt.figure()

# Define the number of colors you want to use
num_colors = 6

# Create a list of colors from a colormap
cmap = sns.color_palette("viridis", num_colors)

# Create a custom discrete color map
cmap_discrete = sns.color_palette(cmap, num_colors).as_hex()

sns.heatmap(dominance / np.sum(dominance, axis=1)[:, None],
            xticklabels=receptor_names, yticklabels=disorders,
            cmap=cmap_discrete, linewidths=.5)
plt.tight_layout()
plt.savefig(path+'figures/schaefer100/heatmap_dominance_enigma_aut_node_IG_6colors.eps')










###NODE GRANULAR ######


autorad_zilles44 = np.load(path+'data/autoradiography/ReceptData_G.npy')
receptor_names = np.load(path+'data/autoradiography/ReceptorNames.npy')
autorad_schaefer100 = make_autorad_schaefer100(autorad_zilles44)
autorad_cammoun033 = make_autorad_cammoun033(autorad_zilles44)
goodidx = np.where(np.isnan(autorad_schaefer100[:, 0]) == False)[0]
badidx = np.where(np.isnan(autorad_schaefer100[:, 0]) == True)[0]
autorad_schaefer100 = zscore(autorad_schaefer100, nan_policy='omit')
nnodes = autorad_schaefer100.shape[0]
autorad_df = pd.DataFrame(data=autorad_schaefer100,
                          index=schaefer['labels'][:50],
                          columns=receptor_names)


ct = np.genfromtxt(path+'data/modified_enigma_atrophy_node.csv', delimiter=',')
disorders = ['node_track', 'node_trackon', 'node_yas']

model_metrics = dict([])
model_pval = np.zeros((len(disorders), ))
perms = np.zeros((33, nspins))
for i in range(nspins):
    perms[:, i] = np.random.permutation(33)
perms = perms.astype(int)

for i in range(len(disorders)):
    print(i)
    m, _ = stats.get_dominance_stats(zscore(autorad_cammoun033),
                                     zscore(ct[:33, i]))
    model_metrics[disorders[i]] = m
    # get model pval
    emp, _ = get_reg_r_sq(zscore(autorad_cammoun033),
                          zscore(ct[:33, i]))
    null = np.zeros((nspins, ))
    for s in range(nspins):
        Xnull = autorad_cammoun033[perms[:, s], :]
        null[s], _ = get_reg_r_sq(zscore(Xnull), zscore(ct[:33, i]))
    model_pval[i] = (1 + sum(null > emp))/(nspins + 1)

dominance = np.zeros((len(disorders), len(receptor_names)))

for i in range(len(model_metrics)):
    tmp = model_metrics[disorders[i]]
    dominance[i, :] = tmp["total_dominance"]
np.save(path+'results/dominance_enigma_autorad_node_G.npy', dominance)

plt.ion()
plt.figure()
plt.bar(np.arange(len(disorders)), np.sum(dominance, axis=1),
        tick_label=disorders)
plt.xticks(rotation='vertical')
plt.tight_layout()
plt.savefig(path+'figures/schaefer100/bar_dominance_enigma_aut_node_G_6colors.eps')

model_pval = multipletests(model_pval, method='fdr_bh')[1]
dominance[np.where(model_pval >= 0.05)[0], :] = 0

plt.ion()
plt.figure()


# Define the number of colors you want to use
num_colors = 6

# Create a list of colors from a colormap
cmap = sns.color_palette("viridis", num_colors)

# Create a custom discrete color map
cmap_discrete = sns.color_palette(cmap, num_colors).as_hex()


sns.heatmap(dominance / np.sum(dominance, axis=1)[:, None],
            xticklabels=receptor_names, yticklabels=disorders,
            cmap=cmap_discrete, linewidths=.5)
plt.tight_layout()
plt.savefig(path+'figures/schaefer100/heatmap_dominance_enigma_aut_node_G_6colors.eps')





###NODE SUPRAGRANULAR ######


autorad_zilles44 = np.load(path+'data/autoradiography/ReceptData_S.npy')
receptor_names = np.load(path+'data/autoradiography/ReceptorNames.npy')
autorad_schaefer100 = make_autorad_schaefer100(autorad_zilles44)
autorad_cammoun033 = make_autorad_cammoun033(autorad_zilles44)
goodidx = np.where(np.isnan(autorad_schaefer100[:, 0]) == False)[0]
badidx = np.where(np.isnan(autorad_schaefer100[:, 0]) == True)[0]
autorad_schaefer100 = zscore(autorad_schaefer100, nan_policy='omit')
nnodes = autorad_schaefer100.shape[0]
autorad_df = pd.DataFrame(data=autorad_schaefer100,
                          index=schaefer['labels'][:50],
                          columns=receptor_names)
ct = np.genfromtxt(path+'data/modified_enigma_atrophy_node.csv', delimiter=',')
disorders = ['node_track', 'node_trackon', 'node_yas']

model_metrics = dict([])
model_pval = np.zeros((len(disorders), ))
perms = np.zeros((33, nspins))
for i in range(nspins):
    perms[:, i] = np.random.permutation(33)
perms = perms.astype(int)

for i in range(len(disorders)):
    print(i)
    m, _ = stats.get_dominance_stats(zscore(autorad_cammoun033),
                                     zscore(ct[:33, i]))
    model_metrics[disorders[i]] = m
    # get model pval
    emp, _ = get_reg_r_sq(zscore(autorad_cammoun033),
                          zscore(ct[:33, i]))
    null = np.zeros((nspins, ))
    for s in range(nspins):
        Xnull = autorad_cammoun033[perms[:, s], :]
        null[s], _ = get_reg_r_sq(zscore(Xnull), zscore(ct[:33, i]))
    model_pval[i] = (1 + sum(null > emp))/(nspins + 1)

dominance = np.zeros((len(disorders), len(receptor_names)))

for i in range(len(model_metrics)):
    tmp = model_metrics[disorders[i]]
    dominance[i, :] = tmp["total_dominance"]
np.save(path+'results/dominance_enigma_autorad_node_SG.npy', dominance)

plt.ion()
plt.figure()
plt.bar(np.arange(len(disorders)), np.sum(dominance, axis=1),
        tick_label=disorders)
plt.xticks(rotation='vertical')
plt.tight_layout()
plt.savefig(path+'figures/schaefer100/bar_dominance_enigma_aut_node_SG_6colors.eps')

model_pval = multipletests(model_pval, method='fdr_bh')[1]
dominance[np.where(model_pval >= 0.05)[0], :] = 0

plt.ion()
plt.figure()
sns.heatmap(dominance / np.sum(dominance, axis=1)[:, None],
            xticklabels=receptor_names, yticklabels=disorders,
            cmap=cmap_discrete, linewidths=.5)
plt.tight_layout()
plt.savefig(path+'figures/schaefer100/heatmap_dominance_enigma_aut_node_SG_6colors.eps')

0
1
2


<ipython-input-14-1b7022590aa9>:71: RuntimeWarning: invalid value encountered in divide
  sns.heatmap(dominance / np.sum(dominance, axis=1)[:, None],


0
1
2


<ipython-input-14-1b7022590aa9>:158: RuntimeWarning: invalid value encountered in divide
  sns.heatmap(dominance / np.sum(dominance, axis=1)[:, None],


0
